In [ ]:
import pandas as pd
from src.config import PROCESSED_DATA_DIR

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

2026-04-23 16:32:07.925 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /Users/spencervenancio/Downloads/STAT 479/SatHealth


In [15]:
df = pd.read_csv(PROCESSED_DATA_DIR / "dataset.csv")
X = df.drop(columns=["depr_prev", "NDVI", "NDVI_binary", "NO2_column_number_density"])
y = df["depr_prev"]  

In [78]:
def corr_screening(df, target_col, selection_criterion = "p_value",
                   corr_threshold=0.2, return_df=False, alpha=0.05, top_K_features=10):
    feature_names = df.drop(columns=[target_col]).columns
    X = df.drop(columns=[target_col]).to_numpy(dtype=float)
    y = df[target_col].to_numpy(dtype=float)
    p = X.shape[1]

    corrs = np.zeros(p)
    pvals = np.zeros(p)
    for j in range(p):
        corrs[j], pvals[j] = stats.pearsonr(X[:, j], y)

    reject, pvals_adj, _, _ = multipletests(pvals, alpha=alpha, method='fdr_bh')
    
    corr_df = pd.DataFrame({
        'feature': feature_names,
        'correlation': corrs,
        'p_value': pvals,
        'p_value_adj': pvals_adj,
        'reject_null': reject
    }).sort_values(by='correlation', key=abs, ascending=False)
    
    if selection_criterion == "correlation":
        selected_features = corr_df.query("abs(correlation) >= @corr_threshold")['feature']
    elif selection_criterion == "p_value":
        selected_features = corr_df.query("reject_null == True")['feature']
    elif selection_criterion == "top_K":
        selected_features = corr_df['feature'].head(top_K_features)

    return corr_df if return_df else selected_features

In [82]:
corr_screening(df, target_col="depr_prev", selection_criterion="p_value")

/var/folders/x5/ktjr6b6s0xg8kqzwxyv56r7m0000gn/T/ipykernel_51200/1123754426.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corrs[j], pvals[j] = stats.pearsonr(X[:, j], y)


1                                                  year
27                                           snow_depth
5               surface_thermal_radiation_downwards_sum
25                                           snow_cover
15                                                ozone
16                              dewpoint_temperature_2m
22                           lake_mix_layer_temperature
17                                       temperature_2m
37                       leaf_area_index_low_vegetation
44                         water-seasonal-coverfraction
26                                         snow_density
18                             soil_temperature_level_1
8     evaporation_from_open_water_surfaces_excluding...
12         total_aerosol_optical_depth_at_550nm_surface
19                             soil_temperature_level_3
30                        volumetric_soil_water_layer_3
23                         lake_total_layer_temperature
29                        volumetric_soil_water_